# 레슨 09 — 시계열 분석 정답지

> 교사·관리자 전용. 학생에게 배포하지 않는다.

이 정답지는 학생용 `mission.md` 의 문제 1~15와 번호가 1:1로 대응한다. 이번 강의의 핵심은 날짜 순서를 기준으로 데이터를 줄이고, 이동평균과 변화율로 흐름을 해석하는 것이다.

## 환경 셀

In [ ]:
import os
import pandas as pd
import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
if IS_COLAB:
    DATA_BASE = "https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-data-analysis/lectures/09/data"
else:
    DATA_BASE = "./data"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

print("pandas:", pd.__version__)
print("data base:", DATA_BASE)

---

## 문제 1 정답 — 일별 지표 데이터 불러오기

In [ ]:
df = pd.read_csv(f"{DATA_BASE}/daily_metrics_2025.csv")

print("shape:", df.shape)
print("columns:", list(df.columns))
print("dtypes:")
print(df.dtypes)
print("앞 5행:")
print(df.head())
print("뒤 3행:")
print(df.tail(3))
print("결측치:")
print(df.isna().sum())
print("날짜 문자열 범위:", df["date"].min(), "~", df["date"].max())

### 왜 이 코드가 정답인지

시계열 분석은 날짜 열이 분석 기준이므로 원본 구조 확인이 먼저다. `shape`, `columns`, `dtypes`, `isna()` 를 확인하면 날짜 변환 전 상태와 숫자 열의 종류를 알 수 있다. 아직 날짜가 문자열이어도 최소값과 최대값을 출력해 데이터 기간을 대략 확인할 수 있다.

**예상 핵심값**

| 항목 | 값 |
|---|---:|
| 행 수 | 365 |
| 열 수 | 6 |
| 결측치 | 0개 |

---

## 문제 2 정답 — 날짜형 변환과 정렬

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
ts = df.set_index("date").sort_index()

print(ts.head())
print("날짜 인덱스 단조 증가:", ts.index.is_monotonic_increasing)
print("날짜 범위:", ts.index.min().date(), "~", ts.index.max().date())
print("전체 일수:", ts.index.nunique())

### 왜 이 코드가 정답인지

`pd.to_datetime` 은 문자열 날짜를 시간 계산이 가능한 타입으로 바꾼다. 날짜 기준 정렬과 인덱스 설정을 해두면 `resample`, `rolling`, `shift`, `pct_change` 를 자연스럽게 사용할 수 있다. 인덱스가 단조 증가인지 확인하면 뒤 계산이 순서 오류 없이 진행되는지 검증할 수 있다.

**예상 핵심값**

```text
전체 일수: 365
```

---

## 문제 3 정답 — 기본 파생 지표 만들기

In [ ]:
ts["conversion_rate"] = ts["orders"] / ts["visitors"]
ts["revenue_per_order"] = ts["revenue"] / ts["orders"]
ts["ad_spend_per_order"] = ts["ad_spend"] / ts["orders"]

print(ts[["visitors", "orders", "revenue", "conversion_rate", "revenue_per_order", "ad_spend_per_order"]].head())
print(ts[["conversion_rate", "revenue_per_order", "ad_spend_per_order"]].agg(["mean", "median"]).round(3))
print("주문 수 최솟값:", ts["orders"].min())

### 왜 이 코드가 정답인지

방문자, 주문, 매출은 규모가 다른 열이므로 비율과 단위당 값을 만들어야 비교하기 쉽다. `conversion_rate` 는 방문이 주문으로 바뀐 비율이고, `revenue_per_order` 는 주문 1건당 매출이다. 주문 수 최솟값을 확인하면 나눗셈에서 0으로 나누는 위험이 있는지 점검할 수 있다.

**채점 기준**

| 파생 열 | 계산식 |
|---|---|
| conversion_rate | orders / visitors |
| revenue_per_order | revenue / orders |
| ad_spend_per_order | ad_spend / orders |

---

## 문제 4 정답 — 월별 리샘플 집계

In [ ]:
monthly = ts.resample("ME").agg(
    visitors=("visitors", "sum"),
    orders=("orders", "sum"),
    revenue=("revenue", "sum"),
    ad_spend=("ad_spend", "sum"),
    temperature_c=("temperature_c", "mean"),
)
monthly["conversion_rate"] = monthly["orders"] / monthly["visitors"]
monthly["revenue_per_order"] = monthly["revenue"] / monthly["orders"]

print(monthly.round(3))
top_month = monthly["revenue"].idxmax()
print("매출 1위 월:", top_month.strftime("%Y-%m"), int(monthly.loc[top_month, "revenue"]))

### 왜 이 코드가 정답인지

`resample("ME")` 는 날짜 인덱스를 월말 기준으로 묶는다. 방문자, 주문, 매출, 광고비는 월별 합계가 필요하고, 기온은 평균이 자연스럽다. 월별 전환율은 일별 전환율 평균보다 월 주문 합계를 월 방문자 합계로 나눈 값이 전체 성과를 더 정확하게 나타낸다.

**예상 확인 포인트**

```text
monthly 행 수: 12개월
```

---

## 문제 5 정답 — 주별 리샘플 집계

In [ ]:
weekly = ts.resample("W").agg(
    visitors=("visitors", "sum"),
    orders=("orders", "sum"),
    revenue=("revenue", "sum"),
    ad_spend=("ad_spend", "sum"),
    temperature_c=("temperature_c", "mean"),
)
weekly["conversion_rate"] = weekly["orders"] / weekly["visitors"]

print(weekly.head(8).round(3))
low_week = weekly["revenue"].idxmin()
print("매출 최저 주 종료일:", low_week.date(), int(weekly.loc[low_week, "revenue"]))

### 왜 이 코드가 정답인지

주별 집계는 월별보다 촘촘해서 단기 흐름을 확인하기 좋다. `resample("W")` 는 주 단위로 데이터를 묶고, 각 주의 종료일을 인덱스로 둔다. 최저 주를 출력하면 단기 하락 구간을 찾아 이후 원인 분석으로 연결할 수 있다.

**주의할 점**

```text
주별 마지막 주는 데이터 기간에 따라 7일보다 짧을 수 있다.
```

---

## 문제 6 정답 — 7일 이동평균

In [ ]:
ts["revenue_7d"] = ts["revenue"].rolling(window=7).mean()
ts["orders_7d"] = ts["orders"].rolling(window=7).mean()
ts["visitors_7d"] = ts["visitors"].rolling(window=7).mean()

print(ts[["revenue", "revenue_7d", "orders", "orders_7d", "visitors", "visitors_7d"]].head(12))
top_7d_day = ts["revenue_7d"].idxmax()
print("7일 매출 이동평균 최고 날짜:", top_7d_day.date(), f"{ts.loc[top_7d_day, 'revenue_7d']:.2f}")

### 왜 이 코드가 정답인지

일별 매출은 변동이 크기 때문에 7일 이동평균으로 흐름을 부드럽게 볼 수 있다. 방문자와 주문에도 같은 기준을 적용하면 세 지표의 추세를 함께 비교할 수 있다. 이동평균 최고 날짜는 단일 최고 매출일보다 안정적인 성과 구간을 찾는 데 유용하다.

**정상 현상**

```text
앞 6행의 이동평균은 NaN 이다.
```

---

## 문제 7 정답 — 14일 이동평균과 추세 비교

In [ ]:
ts["revenue_14d"] = ts["revenue"].rolling(window=14).mean()
ts["orders_14d"] = ts["orders"].rolling(window=14).mean()
ts["revenue_trend_gap"] = ts["revenue_7d"] - ts["revenue_14d"]

print(ts[["revenue", "revenue_7d", "revenue_14d", "revenue_trend_gap"]].tail(10))
print("최근 10일 평균 추세 차이:", f"{ts['revenue_trend_gap'].tail(10).mean():.2f}")

### 왜 이 코드가 정답인지

7일 이동평균은 최근 변화에 민감하고, 14일 이동평균은 더 완만하다. 두 값을 빼면 최근 흐름이 긴 흐름보다 높은지 낮은지 확인할 수 있다. 최근 10일 평균 추세 차이를 보면 마지막 구간이 상승 압력인지 하락 압력인지 간단히 해석할 수 있다.

**해석 기준**

| 값 | 의미 |
|---|---|
| 양수 | 최근 7일 흐름이 14일 흐름보다 높음 |
| 음수 | 최근 7일 흐름이 14일 흐름보다 낮음 |

---

## 문제 8 정답 — 전일 대비 변화량

In [ ]:
ts["revenue_prev_day"] = ts["revenue"].shift(1)
ts["revenue_diff"] = ts["revenue"] - ts["revenue_prev_day"]
ts["orders_diff"] = ts["orders"].diff()

print(ts[["revenue", "revenue_prev_day", "revenue_diff", "orders", "orders_diff"]].head(10))

max_increase_day = ts["revenue_diff"].idxmax()
max_decrease_day = ts["revenue_diff"].idxmin()
print("매출 증가폭 최대 날짜:", max_increase_day.date(), int(ts.loc[max_increase_day, "revenue_diff"]))
print("매출 감소폭 최대 날짜:", max_decrease_day.date(), int(ts.loc[max_decrease_day, "revenue_diff"]))

### 왜 이 코드가 정답인지

`shift(1)` 은 한 행 이전 값을 가져오므로 전날과 현재 값을 비교할 수 있다. `diff()` 는 같은 의미의 변화량 계산을 더 짧게 작성하는 방법이다. 증가폭과 감소폭 최대 날짜를 찾으면 변화가 컸던 시점을 빠르게 확인할 수 있다.

**주의할 점**

```text
첫날은 비교할 전날이 없어 NaN 이 나온다.
```

---

## 문제 9 정답 — 전일 대비 변화율

In [ ]:
ts["revenue_pct_change"] = ts["revenue"].pct_change() * 100
ts["orders_pct_change"] = ts["orders"].pct_change() * 100

print(ts[["revenue", "revenue_pct_change", "orders", "orders_pct_change"]].head(10).round(2))

max_growth_day = ts["revenue_pct_change"].idxmax()
max_drop_day = ts["revenue_pct_change"].idxmin()
print("매출 증가율 최대 날짜:", max_growth_day.date(), f"{ts.loc[max_growth_day, 'revenue_pct_change']:.2f}%")
print("매출 감소율 최대 날짜:", max_drop_day.date(), f"{ts.loc[max_drop_day, 'revenue_pct_change']:.2f}%")

### 왜 이 코드가 정답인지

변화량은 금액 차이를 보여주고, 변화율은 전날 대비 몇 퍼센트 바뀌었는지 보여준다. `pct_change()` 는 이전 행을 기준으로 비율 변화를 계산하므로 시계열에서 전일 대비 해석에 적합하다. 금액 규모가 다른 날끼리 비교할 때는 변화율이 변화량보다 더 공정할 수 있다.

**채점 기준**

| 항목 | 기준 |
|---|---|
| 변화율 | `pct_change() * 100` 사용 |
| 해석 | 증가율과 감소율을 구분 |
| 첫날 | NaN 을 정상으로 이해 |

---

## 문제 10 정답 — 월별 전월 대비 변화율

In [ ]:
monthly["revenue_mom"] = monthly["revenue"].pct_change() * 100
monthly["orders_mom"] = monthly["orders"].pct_change() * 100
monthly["visitors_mom"] = monthly["visitors"].pct_change() * 100

mom_cols = ["revenue", "revenue_mom", "orders", "orders_mom", "visitors", "visitors_mom"]
print(monthly[mom_cols].round(2))

top_mom_month = monthly["revenue_mom"].idxmax()
print("매출 전월 대비 증가율 1위 월:", top_mom_month.strftime("%Y-%m"), f"{monthly.loc[top_mom_month, 'revenue_mom']:.2f}%")

### 왜 이 코드가 정답인지

월별 변화율은 일별 변화율보다 노이즈가 적고 월 단위 운영 판단에 적합하다. 이미 만든 `monthly` 에 `pct_change()` 를 적용하면 전월 대비 변화를 간단히 계산할 수 있다. 첫 월은 비교할 이전 월이 없으므로 NaN 이 나오는 것이 정상이다.

**수업 중 확인 질문**

```text
매출 증가율 1위 월과 매출 금액 1위 월이 같은가?
```

---

## 문제 11 정답 — 요일별 패턴 분석

In [ ]:
ts["weekday"] = ts.index.dayofweek
weekday_summary = ts.groupby("weekday").agg(
    visitors=("visitors", "mean"),
    orders=("orders", "mean"),
    revenue=("revenue", "mean"),
    conversion_rate=("conversion_rate", "mean"),
)

print(weekday_summary.round(3))

fig, ax = plt.subplots(figsize=(6, 3))
sns.barplot(data=weekday_summary.reset_index(), x="weekday", y="revenue", ax=ax)
ax.set_title("Average revenue by weekday")
ax.set_xlabel("Weekday")
ax.set_ylabel("Average revenue")
fig.tight_layout()
plt.close(fig)

print("평균 매출 1위 요일 번호:", int(weekday_summary["revenue"].idxmax()))

### 왜 이 코드가 정답인지

요일별 패턴은 날짜를 요일 번호로 바꾼 뒤 그룹화하면 확인할 수 있다. 일별 합계가 아니라 요일별 평균을 쓰면 각 요일이 여러 번 반복된 결과를 공정하게 비교할 수 있다. 평균 매출 1위 요일을 출력하면 차트 해석을 숫자로 다시 확인할 수 있다.

**주의할 점**

```text
0은 월요일, 6은 일요일이다.
```

---

## 문제 12 정답 — 광고비와 매출의 지연 관계

In [ ]:
ts["ad_spend_lag1"] = ts["ad_spend"].shift(1)
ts["ad_spend_lag7"] = ts["ad_spend"].shift(7)

lag_corr = ts[["revenue", "ad_spend", "ad_spend_lag1", "ad_spend_lag7"]].corr()["revenue"].drop("revenue")
print(lag_corr.round(3))
print("매출과 상관이 가장 큰 광고비 기준:", lag_corr.abs().idxmax(), f"{lag_corr.loc[lag_corr.abs().idxmax()]:.3f}")

### 왜 이 코드가 정답인지

광고비 효과가 당일에만 나타나는지, 며칠 뒤에 나타나는지 가설을 세우려면 지연 열을 만들어 비교할 수 있다. `shift(1)` 과 `shift(7)` 은 각각 1일 전, 7일 전 광고비를 현재 매출과 나란히 놓는다. 상관계수는 관계의 강도를 보는 도구이며, 실제 원인 확인에는 캠페인 일정이나 외부 이벤트 정보가 추가로 필요하다.

**해석 주의**

```text
지연 상관이 크다고 해서 광고비가 매출을 직접 만들었다고 단정하지 않는다.
```

---

## 문제 13 정답 — 이상치 후보 찾기

In [ ]:
ts["revenue_rolling_mean"] = ts["revenue"].rolling(window=14).mean()
ts["revenue_rolling_std"] = ts["revenue"].rolling(window=14).std()
ts["revenue_z"] = (ts["revenue"] - ts["revenue_rolling_mean"]) / ts["revenue_rolling_std"]

high_outliers = ts[ts["revenue_z"] > 2]
low_outliers = ts[ts["revenue_z"] < -2]
outlier_candidates = pd.concat([high_outliers, low_outliers]).sort_index()

print("높은 이상치 후보 수:", len(high_outliers))
print("낮은 이상치 후보 수:", len(low_outliers))
print(outlier_candidates[["revenue", "revenue_rolling_mean", "revenue_z"]].head(10).round(2))

### 왜 이 코드가 정답인지

이동평균은 평소 수준을 나타내고, 이동 표준편차는 평소 변동 폭을 나타낸다. 현재 매출이 평균에서 표준편차 몇 배만큼 벗어났는지 계산하면 갑자기 높거나 낮은 날을 찾을 수 있다. `z` 값 기준은 후보를 찾는 규칙일 뿐, 실제 이상 여부는 원인 확인이 필요하다.

**수업 중 피드백**

```text
이상치 후보는 "삭제할 값"이 아니라 "확인할 값"이라고 말하게 한다.
```

---

## 문제 14 정답 — 누적 지표와 목표 달성일

In [ ]:
ts["cumulative_revenue"] = ts["revenue"].cumsum()
ts["cumulative_orders"] = ts["orders"].cumsum()

fig, ax = plt.subplots(figsize=(8, 3))
sns.lineplot(data=ts.reset_index(), x="date", y="cumulative_revenue", ax=ax)
ax.set_title("Cumulative revenue")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative revenue")
fig.tight_layout()
plt.close(fig)

revenue_half = ts["revenue"].sum() * 0.5
orders_half = ts["orders"].sum() * 0.5
revenue_half_day = ts[ts["cumulative_revenue"] >= revenue_half].index[0]
orders_half_day = ts[ts["cumulative_orders"] >= orders_half].index[0]

print("누적 매출 50% 초과 날짜:", revenue_half_day.date())
print("누적 주문 50% 초과 날짜:", orders_half_day.date())

### 왜 이 코드가 정답인지

누적 지표는 기간 전체 목표가 언제 절반을 넘었는지 같은 질문에 답한다. `cumsum()` 은 날짜 순서대로 값을 더하므로 정렬이 되어 있어야 의미가 있다. 누적 매출 50% 날짜와 누적 주문 50% 날짜를 비교하면 매출과 주문의 진행 속도가 비슷한지 볼 수 있다.

**정상 확인**

```text
누적값은 시간이 지날수록 감소하지 않는다.
```

---

## 문제 15 정답 — 시계열 리포트 결론

In [ ]:
top_month = monthly["revenue"].idxmax()
top_day = ts["revenue"].idxmax()
top_mom_month = monthly["revenue_mom"].idxmax()
top_7d_day = ts["revenue_7d"].idxmax()

print("매출 1위 월:", top_month.strftime("%Y-%m"), int(monthly.loc[top_month, "revenue"]))
print("매출 1위 날짜:", top_day.date(), int(ts.loc[top_day, "revenue"]))
print("전월 대비 증가율 1위 월:", top_mom_month.strftime("%Y-%m"), f"{monthly.loc[top_mom_month, 'revenue_mom']:.2f}%")
print("7일 이동평균 최고 날짜:", top_7d_day.date(), f"{ts.loc[top_7d_day, 'revenue_7d']:.2f}")
print("이상치 후보 수:", len(outlier_candidates))

print("결론 초안:")
print(f"- 월별 매출은 {top_month.strftime('%Y-%m')}에 가장 높았다.")
print(f"- 단일 날짜 기준 최고 매출일은 {top_day.date()}이다.")
print(f"- 전월 대비 증가율은 {top_mom_month.strftime('%Y-%m')}에 가장 컸다.")
print("- 다음 분석에서는 이상치 후보 날짜의 캠페인, 휴일, 외부 이벤트를 확인한다.")

### 왜 이 코드가 정답인지

시계열 결론은 월별 흐름, 단일 최고일, 변화율, 이동평균, 이상치 후보를 함께 봐야 한다. 한 지표만 보면 일시적인 큰 값에 끌릴 수 있지만 여러 기준을 같이 출력하면 흐름과 이벤트를 구분할 수 있다. 마지막 문장에 추가 확인 행동을 넣으면 분석 결과가 다음 운영 판단으로 이어진다.

**모범 결론 예시**

```text
월별 매출은 특정 월에 가장 높았고, 전월 대비 증가율이 큰 월은 별도로 확인해야 한다.
7일 이동평균 최고일은 단일 최고 매출일보다 안정적인 성과 구간을 보여준다.
이상치 후보 날짜는 오류가 아니라 캠페인이나 휴일 효과일 수 있다.
다음 분석에서는 광고비 지연 상관과 이벤트 캘린더를 함께 비교한다.
```

---

## 채점 포인트

| 항목 | 확인 기준 |
|---|---|
| 날짜 처리 | `to_datetime`, 정렬, 날짜 인덱스 설정 |
| 리샘플 | 월별·주별 집계 기준이 명확함 |
| 이동평균 | 7일·14일 창을 구분해 사용 |
| 변화율 | 변화량과 변화율을 혼동하지 않음 |
| 결론 | 최고 시점, 변화율, 이상치 후보, 다음 행동 포함 |

## 흔한 오답

- 날짜형 변환 전에 `resample` 을 사용한다.
- 정렬하지 않은 상태에서 `shift` 와 `rolling` 을 적용한다.
- 월별 전환율을 일별 전환율 평균으로만 해석한다.
- 변화량과 변화율을 같은 의미로 쓴다.
- 이상치 후보를 무조건 제거해야 하는 값으로 설명한다.

## 교사용 상세 피드백 가이드

학생 답안을 볼 때는 계산 결과보다 날짜 기준을 먼저 본다. 다음 순서로 확인한다.

1. **날짜 처리 순서가 맞는가**: `to_datetime`, 정렬, 인덱스 설정 중 하나라도 빠지면 뒤 계산이 실행되어도 해석이 흔들린다.
2. **집계 기준을 설명할 수 있는가**: 월별, 주별, 일별 중 어떤 질문에 어떤 단위를 쓰는지 말하게 한다.
3. **이동평균을 원본처럼 말하지 않는가**: 이동평균은 추세를 보기 위한 보조 지표다. 특정 날짜의 실제 매출과 다르다는 점을 확인한다.
4. **변화율과 변화량을 구분하는가**: 금액 차이와 퍼센트 차이는 상황에 따라 다른 결론을 만든다.

## 부분 점수 기준

| 상황 | 처리 |
|---|---|
| 파일 로드만 성공 | 문제 1 일부 통과 |
| 날짜 변환은 했지만 정렬 누락 | 코드 보완 필요 |
| 월별 집계는 맞지만 파생 지표 없음 | 집계 일부 통과, 해석 보충 |
| 이동평균 NaN 을 오류로 지움 | 개념 설명 후 수정 |
| 결론이 숫자와 맞지 않음 | 결론 재제출 |

## 보너스 확장 아이디어

빠른 학생에게는 휴일 또는 캠페인 플래그를 가정해 추가 분석하게 한다. 예를 들어 주말 여부, 광고비 상위 20% 여부, 기온 구간, 전환율 상위 날짜를 만들어 월별 매출 흐름과 함께 비교할 수 있다. 단, 기준을 추가할수록 결론이 복잡해지므로 최종 리포트에는 가장 설명력이 큰 기준만 남기게 한다.

## 수업 중 피드백 문장 예시

- “이 값은 실제 매출인가요, 이동평균인가요?”
- “첫 행의 변화율이 비어 있는 이유를 설명해 봅시다.”
- “월별 전환율은 주문 합계와 방문자 합계로 다시 계산해 보세요.”
- “이상치 후보를 삭제하기 전에 어떤 일이 있었는지 확인해야 합니다.”

## 모범 답안 사용 주의

모범 답안은 날짜 처리와 해석 기준을 맞추기 위한 예시다. 학생이 `date` 를 열로 유지한 채 `groupby(pd.Grouper(...))` 를 사용해도 같은 질문에 답하면 통과시킬 수 있다. 다만 정렬 없이 변화량을 계산했거나, 결론이 출력 숫자와 맞지 않으면 실행 여부와 별개로 보충이 필요하다.

## 재현성 확인

강사는 답안 노트북을 런타임 재시작 후 처음부터 실행해 본다. 시계열 노트북은 앞 셀에서 만든 `ts`, `monthly`, `weekly` 에 많이 의존하므로 셀 순서가 바뀌면 실패할 수 있다. 특히 `pct_change`, `rolling`, `shift` 는 날짜 정렬 이후 실행되는지 확인한다.

## 결론 문장 채점 예시

결론이 “매출이 증가했다”에서 끝나면 부족하다. “월별 매출은 3월에 가장 높았고, 2월 대비 증가율도 커서 캠페인 여부를 확인해야 한다”처럼 기간, 지표, 비교, 다음 행동이 함께 들어가야 한다.

## 교사용 심화 채점 메모

시계열 수업에서 가장 자주 생기는 문제는 문법 오류가 아니라 기준 오류다. 학생 코드가 실행되어도 날짜 정렬이 빠져 있으면 `shift`, `diff`, `pct_change`, `rolling` 의 결과가 모두 다른 의미가 된다. 채점할 때는 셀별 출력보다 먼저 `date` 변환, 정렬, 인덱스 설정 순서가 보이는지 확인한다.

| 구간 | 반드시 볼 기준 |
|---|---|
| 문제 1~2 | 날짜형 변환과 정렬이 먼저 이루어졌는가 |
| 문제 3~5 | 합계 지표와 평균 지표를 구분했는가 |
| 문제 6~7 | 이동평균 NaN 을 정상으로 이해하는가 |
| 문제 8~10 | 변화량과 변화율을 다른 말로 설명하는가 |
| 문제 11~12 | 요일과 지연 상관을 보조 해석으로만 쓰는가 |
| 문제 13~15 | 이상치 후보와 결론이 실제 숫자에 연결되는가 |

## 추가 피드백 예시

- “지금 결과는 맞아 보이지만 날짜 순서가 보장되어 있나요?”
- “전환율은 일별 평균을 다시 평균 낸 값과 월 주문 합계를 월 방문자 합계로 나눈 값이 다를 수 있습니다.”
- “이동평균 최고일과 실제 매출 최고일이 다른 이유를 설명해 봅시다.”
- “변화량이 큰 날과 변화율이 큰 날은 다를 수 있습니다.”
- “이상치 후보를 발견했다면 삭제가 아니라 원인 확인이 먼저입니다.”

## 수업 운영 팁

빠른 학생에게는 `resample("QE")` 로 분기별 집계를 추가하게 한다. 중간 속도의 학생에게는 월별 집계와 7일 이동평균까지만 정확히 끝내게 한다. 느린 학생에게는 문제 1~4를 안정적으로 완료하게 하고, 변화율과 이상치 문제는 교사가 함께 코드 흐름을 따라간다.

시계열 수업에서는 그래프가 없어도 표 해석이 가능해야 한다. 학생이 차트만 보고 결론을 쓰면 월별 표와 변화율 표를 다시 출력하게 한다. 반대로 표만 있고 결론이 없는 학생에게는 최고 월, 증가율 1위 월, 이상치 후보 수를 사용해 3문장 리포트를 먼저 쓰게 한다.

## 확장 과제 기준

심화 과제로는 아래 중 하나를 선택하게 한다.

1. 광고비 상위 20% 날짜와 하위 20% 날짜의 매출 평균 비교
2. 주말과 평일의 전환율 비교
3. 기온 구간별 방문자 수 비교
4. 월별 광고비 대비 매출 비율 계산
5. 이동평균 교차 지점을 사용한 상승·하락 구간 표시

확장 과제도 원칙은 같다. 기준을 새로 만들면 반드시 그 기준이 어떤 질문에 답하는지 쓰게 한다. 예를 들어 광고비 상위 20%를 만들었다면 "광고비를 많이 쓴 날 매출이 실제로 높았는가"라는 질문까지 함께 제시해야 한다.

## 재제출 안내 기준

다음 상황은 코드가 실행되어도 재제출을 권한다.

- 날짜 정렬 없이 변화량을 계산했다.
- 월별 합계를 내야 하는데 월별 평균을 사용했다.
- 변화율 단위를 소수와 퍼센트로 섞어 썼다.
- 이상치 후보를 데이터 오류로 단정했다.
- 결론에 날짜나 기간이 없다.

반대로 변수명이나 차트 스타일이 모범 답안과 달라도 기준이 정확하고 결론이 출력과 맞으면 통과시킨다. 이 강의의 목적은 정답 코드를 외우는 것이 아니라 날짜 기준의 사고방식을 익히는 것이다.